# Ch.4 — Small Steps on a Curve

**Running theme.** Fixed release speed 8 m/s. Which release angle $\theta$ maximises the horizontal range $R(\theta) = (v_0^2/g)\sin(2\theta)$? We know the answer analytically (45°), but we want an algorithm that *finds* it by walking uphill — because the moment we add wind or drag, the analytic answer vanishes but the algorithm still works.

**Learning outcomes.**
1. Run gradient ascent on $R(\theta)$ and watch it converge — or not — depending on the step size.
2. Drag a step-size slider and *see* divergence happen live.
3. Meet a non-convex landscape and understand why initialisation matters.
4. Sweep starting angles to map the *basin of attraction* of each optimum.

## 1 · Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import (FloatSlider, FloatText, FloatLogSlider, IntSlider,
                         HBox, VBox, Output, jslink)

try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    get_ipython().run_line_magic("matplotlib", "inline")

plt.rcParams.update({"figure.figsize": (8.5, 5.0), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25})

def linked_pair(label, value, vmin, vmax, step=0.05):
    sl = FloatSlider(value=value, min=vmin, max=vmax, step=step,
                     description=label, continuous_update=True, readout=False)
    tx = FloatText(value=value, description=" ", step=step,
                   layout={"width": "110px"})
    jslink((sl, "value"), (tx, "value"))
    return sl, tx

# Free-throw parameters ---------------------------------------------------
v0, g = 8.0, 9.81

def R(theta_deg):
    return v0 ** 2 / g * np.sin(2 * np.radians(theta_deg))

def dR(theta_deg):
    # d R / d theta_deg = (2 v0^2/g) cos(2θ) · (π/180)
    return 2 * v0 ** 2 / g * np.cos(2 * np.radians(theta_deg)) * (np.pi / 180)

## 2 · Six lines of Python — the whole algorithm

Gradient ascent (we are maximising) from an arbitrary start.

In [ ]:
def ascend(start_deg, eta, tol=1e-6, max_iter=500):
    theta = float(start_deg)
    history = [theta]
    for _ in range(max_iter):
        grad = dR(theta)
        if abs(grad) < tol:
            break
        theta = theta + eta * grad    # + because we want to MAXIMISE
        history.append(theta)
    return np.array(history)

for start in [10, 25, 60, 80]:
    path = ascend(start_deg=start, eta=35.0)
    print(f"start = {start:>3}°   final = {path[-1]:6.3f}°   steps = {len(path)-1:>3}   R_final = {R(path[-1]):.4f} m")

## 3 · Interactive — starting angle + step size

Drag the two sliders. The coloured trajectory shows each iterate landing on the curve; the text-box readout tracks the final angle and range.

In [ ]:
start_sl, start_tx = linked_pair("θ₀ (deg)", value=20.0, vmin=5.0, vmax=85.0, step=1.0)
eta_sl = FloatLogSlider(value=35.0, base=10, min=-1, max=2.5, step=0.05,
                        description="step η", readout=True, readout_format=".2f")
steps_sl = IntSlider(value=15, min=1, max=60, step=1, description="# iters",
                     continuous_update=True)
trace_out = Output()

theta_grid = np.linspace(5, 85, 400)
R_grid = R(theta_grid)

fig, ax = plt.subplots()
ax.plot(theta_grid, R_grid, color="#2E86C1", lw=2.2, label="R(θ)")
ax.axvline(45, color="#27AE60", lw=1.0, linestyle=":", alpha=0.7)
ax.scatter([45], [R(45)], color="#27AE60", s=60, zorder=5, label="true optimum")
traj_line, = ax.plot([], [], color="#E67E22", lw=1.4, marker="o", markersize=4,
                     alpha=0.9, label="gradient ascent")
ax.set_xlabel("release angle  θ  (degrees)"); ax.set_ylabel("range  R  (m)")
ax.set_xlim(0, 90); ax.set_ylim(0, R(45) * 1.15); ax.legend(loc="lower center")

def redraw(*_):
    path = ascend(start_sl.value, eta_sl.value, max_iter=steps_sl.value)
    path = np.clip(path, 0.1, 89.9)
    traj_line.set_data(path, R(path))
    with trace_out:
        trace_out.clear_output(wait=True)
        print(f"iters run : {len(path) - 1}")
        print(f"final θ   : {path[-1]:.4f}°")
        print(f"final R   : {R(path[-1]):.4f} m")
        print(f"|θ - 45|  : {abs(path[-1] - 45):.4f}°")
    fig.canvas.draw_idle()

for w in (start_sl, eta_sl, steps_sl): w.observe(redraw, names="value")
redraw()

display(VBox([HBox([start_sl, start_tx]), eta_sl, steps_sl, trace_out]))

**Things to try:**

1. Start at 20°, $\eta = 10$. Watch it crawl.
2. Start at 20°, $\eta = 35$. Clean convergence in a dozen steps.
3. Start at 20°, $\eta = 180$. Overshoot past 45°, ping-pong.
4. Start at 20°, $\eta = 300$. **Diverges** — iterates leave the [0°, 90°] range.
5. Set steps to 1 and drag $\eta$ to *see* the first step itself grow with $\eta$.

## 4 · A systematic look — error vs iteration for four step sizes

Plot $|θ_k - 45°|$ on a log-y axis. Linear-looking lines on a log plot = exponential convergence.

In [ ]:
fig2, ax2 = plt.subplots()
for eta_val, col, lbl in [(5,    "#E74C3C", "η = 5    (too small)"),
                           (35,   "#27AE60", "η = 35   (just right)"),
                           (160,  "#E67E22", "η = 160  (oscillating)"),
                           (300,  "#7F8C8D", "η = 300  (diverges)")]:
    path = ascend(start_deg=20.0, eta=eta_val, max_iter=40, tol=0)
    err = np.abs(path - 45.0)
    # clip to plottable range
    err = np.where(err > 1e-10, err, 1e-10)
    ax2.semilogy(err, color=col, lw=1.6, marker="o", markersize=3.5, label=lbl)
ax2.set_xlabel("iteration  k"); ax2.set_ylabel("|θ_k − 45°|")
ax2.set_title("Convergence behaviour on a log scale")
ax2.legend(loc="upper right"); ax2.grid(True, alpha=0.3)
plt.show()

## 5 · Non-convex landscape — starting angle decides your fate

Add a crosswind that penalises some angles and rewards others. The curve grows a second hump; the starting angle now decides whether you find the true best release or settle for a mediocre one.

In [ ]:
def R_windy(theta_deg):
    base = R(theta_deg)
    bump = 1.2 * np.exp(-((theta_deg - 60) / 8) ** 2)
    bonus = 0.9 * np.exp(-((theta_deg - 30) / 6) ** 2)
    return base - bump + bonus

def dR_windy(theta_deg, h=0.05):
    return (R_windy(theta_deg + h) - R_windy(theta_deg - h)) / (2 * h)

def ascend_bumpy(start_deg, eta, max_iter=60, tol=1e-6):
    theta = float(start_deg)
    path = [theta]
    for _ in range(max_iter):
        g_k = dR_windy(theta)
        if abs(g_k) < tol: break
        theta = np.clip(theta + eta * g_k, 5, 85)
        path.append(theta)
    return np.array(path)

theta_grid = np.linspace(5, 85, 400)

start_bumpy_sl, start_bumpy_tx = linked_pair("θ₀ (deg)", 50.0, 5.0, 85.0, step=1.0)
out_bumpy = Output()

fig3, ax3 = plt.subplots()
ax3.plot(theta_grid, R_windy(theta_grid), color="#8E44AD", lw=2.2, label="R_wind(θ)")
global_theta = theta_grid[int(np.argmax(R_windy(theta_grid)))]
ax3.scatter([global_theta], [R_windy(global_theta)], color="#27AE60", s=60,
            zorder=5, label=f"global optimum ≈ {global_theta:.1f}°")
bumpy_line, = ax3.plot([], [], color="#E67E22", lw=1.4, marker="o", markersize=3.5,
                       alpha=0.9, label="your trajectory")
ax3.set_xlabel("θ (deg)"); ax3.set_ylabel("R_wind (m)")
ax3.legend(loc="lower center")

def redraw_bumpy(*_):
    path = ascend_bumpy(start_bumpy_sl.value, eta=5.0)
    bumpy_line.set_data(path, R_windy(path))
    with out_bumpy:
        out_bumpy.clear_output(wait=True)
        print(f"start θ₀   : {start_bumpy_sl.value:.1f}°")
        print(f"final θ    : {path[-1]:.3f}°")
        print(f"final R    : {R_windy(path[-1]):.4f} m")
        print(f"global R   : {R_windy(global_theta):.4f} m")
        gap = R_windy(global_theta) - R_windy(path[-1])
        print(f"optimality gap: {gap:.4f} m  → {'OK' if gap < 0.05 else 'SUBOPTIMAL'}")
    fig3.canvas.draw_idle()

start_bumpy_sl.observe(redraw_bumpy, names="value")
redraw_bumpy()
display(VBox([HBox([start_bumpy_sl, start_bumpy_tx]), out_bumpy]))

## 6 · Basin-of-attraction plot — the unfairness of initialisation

Sweep starting angles from 5° to 85° and record the final $R$ each one converges to. The result is a staircase: the flat regions are the basins of attraction — the set of starts that all end up at the same local optimum.

In [ ]:
starts = np.arange(6, 85, 1.0)
finals = np.array([ascend_bumpy(s, eta=5.0)[-1] for s in starts])
R_finals = R_windy(finals)

fig4, (axA, axB) = plt.subplots(1, 2, figsize=(11.5, 4.5))
axA.plot(theta_grid, R_windy(theta_grid), color="#8E44AD", lw=1.8, alpha=0.6)
axA.scatter(finals, R_windy(finals), color="#E67E22", s=18, alpha=0.8,
            label="landing point per start")
axA.axvline(global_theta, color="#27AE60", lw=1.0, linestyle=":")
axA.set_xlabel("final θ"); axA.set_ylabel("final R"); axA.legend()
axA.set_title("Where does each start land?")

axB.plot(starts, R_finals, color="#E67E22", lw=1.6, marker="o", markersize=3)
axB.axhline(R_windy(global_theta), color="#27AE60", lw=1.0, linestyle=":",
            label="global best")
axB.set_xlabel("starting angle θ₀"); axB.set_ylabel("final R reached")
axB.set_title("Basin of attraction — the staircase")
axB.legend()
plt.tight_layout(); plt.show()

# Summary: fraction of starts that reach the global optimum
frac = float(np.mean(R_finals > R_windy(global_theta) - 0.05))
print(f"{frac:.0%} of starts land within 0.05 m of the global optimum.")

## 7 · Where this reappears

- **Pre-Req Ch.6.** The scalar update $\theta \leftarrow \theta + \eta\,R'(\theta)$ becomes the vector update $\boldsymbol\theta \leftarrow \boldsymbol\theta - \eta\,\nabla L(\boldsymbol\theta)$. Same recipe, higher dimension.
- **ML Ch.1 / Ch.5.** Every optimiser in every training loop is this algorithm. Adam, RMSProp, and SGD+momentum are patches that address the problems you just saw.
- **Non-convexity is the story of deep learning.** The right-panel plot is why initialisation schemes, warm restarts, and ensembles exist — we cannot defeat non-convexity, only survive it.

**Final exercise.** Modify `ascend_bumpy` to add momentum: $v_{k+1} = \mu\,v_k + g_k$, then $\theta_{k+1} = \theta_k + \eta\,v_{k+1}$. Try $\mu = 0.9$. Does the fraction of starts that find the global optimum change? (Expected: momentum helps roll over shallow dips — some previously-stuck starts now escape.)